# 02 – Seleção e limpeza

**Objetivo:** aplicar os tratamentos definidos no diagnóstico do notebook 01, por meio das funções
documentadas em `src/funcoes.py`. Cada etapa mostra o estado dos dados antes e depois do tratamento.
Ao final, o dataset limpo é salvo para as análises do notebook 03.

In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from src.funcoes import (
    carregar_dados, selecionar_colunas, renomear_colunas,
    converter_para_numerico, preencher_faltantes, marcar_como_faltante,
    remover_duplicatas, criar_coluna_data, limpar_dados, salvar_dados,
    corrigir_valores,
)

pd.set_option("display.max_columns", None)

In [2]:
df_bruto = carregar_dados("../data/spotify-2023.csv")

Dados carregados: 953 linhas e 24 colunas.


## Etapa 1 – Seleção de colunas
Mantive apenas as 19 colunas relacionadas ao Spotify. As colunas de Apple Music,
Deezer e Shazam ficam fora do escopo; com isso, também saem `in_deezer_playlists` e
`in_shazam_charts`, que tinham tipos incorretos e valores faltantes.

In [3]:
colunas_spotify = [
    "track_name", "artist(s)_name", "artist_count",
    "released_year", "released_month", "released_day",
    "streams", "in_spotify_playlists", "in_spotify_charts",
    "bpm", "key", "mode",
    "danceability_%", "valence_%", "energy_%", "acousticness_%",
    "instrumentalness_%", "liveness_%", "speechiness_%",
]

df = selecionar_colunas(df_bruto, colunas_spotify)
print(f"Antes: {df_bruto.shape} | Depois: {df.shape}")

Antes: (953, 24) | Depois: (953, 19)


In [4]:
try:
    selecionar_colunas(df_bruto, ["track_name", "coluna_que_nao_existe"])
    print("FALHOU: a função deveria ter levantado um erro.")
except KeyError as erro:
    print(f"Teste OK – erro capturado: {erro}")

Teste OK – erro capturado: "Colunas não encontradas no DataFrame: ['coluna_que_nao_existe']"


## Etapa 2 – Renomeação de colunas
Removi símbolos dos nomes (`(s)`, `%`), que dificultam a escrita do código.
As demais colunas já estão em um formato adequado e mantêm o nome original.

In [5]:
novos_nomes = {
    "artist(s)_name": "artist_name",
    "danceability_%": "danceability",
    "valence_%": "valence",
    "energy_%": "energy",
    "acousticness_%": "acousticness",
    "instrumentalness_%": "instrumentalness",
    "liveness_%": "liveness",
    "speechiness_%": "speechiness",
}

df = renomear_colunas(df, novos_nomes)
df.columns.tolist()

['track_name',
 'artist_name',
 'artist_count',
 'released_year',
 'released_month',
 'released_day',
 'streams',
 'in_spotify_playlists',
 'in_spotify_charts',
 'bpm',
 'key',
 'mode',
 'danceability',
 'valence',
 'energy',
 'acousticness',
 'instrumentalness',
 'liveness',
 'speechiness']

## Etapa 3 – Conversão de `streams` para número
`streams` foi lida como texto por causa de um único valor corrompido (linha 574, "Love Grows").
Esse valor vira `NaN`; a linha é mantida, pois as demais colunas estão normais.

In [6]:
print(f"Tipo antes: {df['streams'].dtype}")
print(f"Linha 574 antes: {df.loc[574, 'streams']}\n")

df = converter_para_numerico(df, "streams")

print(f"\nTipo depois: {df['streams'].dtype}")
print(f"Linha 574 depois: {df.loc[574, 'streams']}")

Tipo antes: str
Linha 574 antes: BPM110KeyAModeMajorDanceability53Valence75Energy69Acousticness7Instrumentalness0Liveness17Speechiness3

Aviso: 1 valor(es) de 'streams' não eram numéricos e viraram NaN.

Tipo depois: float64
Linha 574 depois: nan


In [7]:
# Checagem

df["streams"].describe()

count    9.520000e+02
mean     5.141374e+08
std      5.668569e+08
min      2.762000e+03
25%      1.416362e+08
50%      2.905309e+08
75%      6.738690e+08
max      3.703895e+09
Name: streams, dtype: float64

**Conferindo:** O valor mínimo de 2762 streams é estranho, considerando que o primeiro quartil é de 141636175.0. Dado isso, vou conferir os 5 menores valores:

In [8]:
colunas_ver = ["track_name", "artist_name", "released_year", "released_month",
               "released_day", "streams", "in_spotify_playlists", "in_spotify_charts"]
df.nsmallest(5, "streams")[colunas_ver]

,track_name,artist_name,released_year,released_month,released_day,streams,in_spotify_playlists,in_spotify_charts
123,Que Vuelvas,"Carin Leon, Grupo Frontera",2022,12,9,2762.0,763,26
393,Jhoome Jo Pathaan,"Arijit Singh, Vishal Dadlani, Sukriti Kakar, V...",2022,12,22,1365184.0,138,4
144,QUEMA,"Sog, Ryan Castro, Peso Pluma",2023,7,13,11599388.0,437,31
142,"Gol Bolinha, Gol Quadrado 2","Mc Pedrinho, DJ 900",2023,6,1,11956641.0,293,8
68,Overdrive,Post Malone,2023,7,14,14780425.0,410,36


### Valor inválido em `streams`: "Que Vuelvas"
Após a conversão, o mínimo de `streams` é 2.762, cerca de 500 vezes menor que o segundo menor
valor (1,37 mi). O valor é incoerente com as demais medidas de sucesso da música (763 playlists,
presença nos charts) e com a data de lançamento (09/12/2022, meses antes da coleta). Conclusão:
o valor pode ser um erro de coleta. O valor vira `NaN` e a linha é mantida.

In [9]:
erro_streams = (
    (df["track_name"] == "Que Vuelvas")
    & (df["artist_name"] == "Carin Leon, Grupo Frontera")
)

df = marcar_como_faltante(df, "streams", erro_streams,
                          motivo="Erro de coleta: 2.762 streams, incoerente com playlists e charts.")

df.nsmallest(3, "streams")[colunas_ver]

1 valor(es) de 'streams' marcado(s) como faltante(s). Erro de coleta: 2.762 streams, incoerente com playlists e charts.


,track_name,artist_name,released_year,released_month,released_day,streams,in_spotify_playlists,in_spotify_charts
393,Jhoome Jo Pathaan,"Arijit Singh, Vishal Dadlani, Sukriti Kakar, V...",2022,12,22,1365184.0,138,4
144,QUEMA,"Sog, Ryan Castro, Peso Pluma",2023,7,13,11599388.0,437,31
142,"Gol Bolinha, Gol Quadrado 2","Mc Pedrinho, DJ 900",2023,6,1,11956641.0,293,8


In [10]:
# Checagem

df["streams"].count()

np.int64(951)

## Etapa 4 – Preenchimento dos faltantes de `key`
Os 95 fvalores faltantes da coluna `key` correspondem à nota C, o que foi conferido a partir de uma amostra
aleatória de 10 músicas: (10 de 10 em C). Por isso, optei por preencher com "C", registrando essa suposição.

In [11]:
antes = df["key"].value_counts(dropna=False)

df = preencher_faltantes(df, "key", "C")

depois = df["key"].value_counts(dropna=False)
pd.DataFrame({"antes": antes, "depois": depois}).fillna(0).astype(int)

95 valor(es) faltante(s) em 'key' preenchido(s) com 'C'.


,antes,depois
key,,
A,75,75
A#,57,57
B,81,81
C,0,95
C#,120,120
D,81,81
D#,33,33
E,62,62
F,89,89


## Etapa 5 – Remoção de duplicatas
Quatro músicas aparecem duas vezes (mesmo nome e artista). Decisão:
manter, em cada par, a linha com mais `in_spotify_playlists`, evitando contar o mesmo sucesso duas
vezes. Nos casos de versões diferentes as características de áudio mantidas
podem não representar todos os streams (é uma limitação para análise geral, mas são poucos casos).

In [12]:
chave = ["track_name", "artist_name"]
colunas_dup = chave + ["streams", "in_spotify_playlists"]

print("ANTES:")
display(df[df.duplicated(subset=chave, keep=False)].sort_values(chave)[colunas_dup])

df = remover_duplicatas(df, chave, "in_spotify_playlists")

print("DEPOIS:")
musicas_dup = ["About Damn Time", "SNAP", "SPIT IN MY FACE!", "Take My Breath"]
df[df["track_name"].isin(musicas_dup)][colunas_dup]

ANTES:


,track_name,artist_name,streams,in_spotify_playlists
372,About Damn Time,Lizzo,723894473.0,2332
764,About Damn Time,Lizzo,723894473.0,9021
178,SNAP,Rosa Linn,726307468.0,3202
873,SNAP,Rosa Linn,711366595.0,1818
345,SPIT IN MY FACE!,ThxSoMch,303216294.0,629
482,SPIT IN MY FACE!,ThxSoMch,301869854.0,573
512,Take My Breath,The Weeknd,130655803.0,2597
616,Take My Breath,The Weeknd,432702334.0,6392


4 linha(s) duplicada(s) removida(s) (mantida a de maior 'in_spotify_playlists').
DEPOIS:


,track_name,artist_name,streams,in_spotify_playlists
178,SNAP,Rosa Linn,726307468.0,3202
345,SPIT IN MY FACE!,ThxSoMch,303216294.0,629
616,Take My Breath,The Weeknd,432702334.0,6392
764,About Damn Time,Lizzo,723894473.0,9021


In [13]:
# Verificando

print(f"Formato atual: {df.shape}")

assert df.duplicated(subset=chave).sum() == 0, "Ainda há duplicatas"
assert len(df) == 949, "Deveriam restar 949 linhas (953 - 4)"
print("Verificações OK")

Formato atual: (949, 19)
Verificações OK


In [14]:
df[['released_year', 'track_name', 'artist_name']].sort_values(by='released_year', ascending=True)

,released_year,track_name,artist_name
439,1930,Agudo M��gi,"Styrx, utku INC, Thezth"
469,1942,White Christmas,"Bing Crosby, John Scott Trotter & His Orchestr..."
460,1946,The Christmas Song (Merry Christmas To You) - ...,Nat King Cole
466,1950,Let It Snow! Let It Snow! Let It Snow!,"Frank Sinatra, B. Swanson Quartet"
459,1952,A Holly Jolly Christmas - Single Version,Burl Ives
...,...,...,...
30,2023,Rush,Troye Sivan
29,2023,Dance The Night (From Barbie The Album),Dua Lipa
0,2023,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook"
28,2023,Last Night,Morgan Wallen


## Etapa 6 – Tratamento das datas
1. **Ano incorreto:** "Agudo Mági" (Styrx, utku INC, Thezth) consta como de 1930, o que com certeza está incorreto, pois conheço a música. Como se trata de apenas uma música, é fácil de pesquisar o valor real do ano de lançamento.
2. **Coluna `data_lancamento`:** data completa em formato datetime, para facilitar as análises
   temporais.
3. **Datas 01/01:** 65 músicas com indício de data de preenchimento. Não serão apagadas, mas marcadas
   na coluna `data_suspeita`, nas análises decidirei o que fazer caso a caso.

In [15]:
ano_errado = (
    df["track_name"].str.startswith("Agudo") # Usei o starswith pra localizar, pois o nome real tava com caracteres perdidos
    & df["artist_name"].str.contains("Styrx", regex=False)
)

fonte = "Fonte: página da faixa no Spotify."

df = corrigir_valores(df, "released_year", ano_errado, 2022, motivo=fonte)
df = corrigir_valores(df, "released_month", ano_errado, 10, motivo=fonte)
df = corrigir_valores(df, "released_day", ano_errado, 9, motivo=fonte)

1 valor(es) de 'released_year' corrigido(s) para 2022. Fonte: página da faixa no Spotify.
1 valor(es) de 'released_month' corrigido(s) para 10. Fonte: página da faixa no Spotify.
1 valor(es) de 'released_day' corrigido(s) para 9. Fonte: página da faixa no Spotify.


In [16]:
# Criando coluna de data:

df = criar_coluna_data(df, "released_year", "released_month", "released_day")

df[["track_name", "released_year", "released_month", "released_day", "data_lancamento"]].head()

,track_name,released_year,released_month,released_day,data_lancamento
0,Seven (feat. Latto) (Explicit Ver.),2023,7,14,2023-07-14
1,LALA,2023,3,23,2023-03-23
2,vampire,2023,6,30,2023-06-30
3,Cruel Summer,2019,8,23,2019-08-23
4,WHERE SHE GOES,2023,5,18,2023-05-18


**Explicando o aviso:** A data que não pode ser montada foi a do Agudo Mágico, música que teve o released_year invalidado anteriormente.

In [17]:
# Marcação das músicas com data 01/01:

df["data_suspeita"] = (df["released_month"] == 1) & (df["released_day"] == 1)

print(f"Datas marcadas como suspeitas: {df['data_suspeita'].sum()}")
print(f"Suspeitas entre 2022 e 2023: {(df['data_suspeita'] & (df['released_year'] >= 2022)).sum()}")

Datas marcadas como suspeitas: 64
Suspeitas entre 2022 e 2023: 0


## Etapa 7 – Função de limpeza completa
A função `limpar_dados` (em `src/funcoes.py`) aplica todas as etapas anteriores, na mesma ordem,
a partir dos dados brutos. 

In [18]:
df_pipeline = limpar_dados(df_bruto)

pd.testing.assert_frame_equal(df_pipeline, df)
print("\nOK: limpar_dados reproduz exatamente o passo a passo.")

Aviso: 1 valor(es) de 'streams' não eram numéricos e viraram NaN.
1 valor(es) de 'streams' marcado(s) como faltante(s). Erro de coleta (Que Vuelvas).
95 valor(es) faltante(s) em 'key' preenchido(s) com 'C'.
4 linha(s) duplicada(s) removida(s) (mantida a de maior 'in_spotify_playlists').
1 valor(es) de 'released_year' corrigido(s) para 2022. Fonte: página da faixa no Spotify.
1 valor(es) de 'released_month' corrigido(s) para 10. Fonte: página da faixa no Spotify.
1 valor(es) de 'released_day' corrigido(s) para 9. Fonte: página da faixa no Spotify.

OK: limpar_dados reproduz exatamente o passo a passo.


## Resultado final: antes x depois

In [19]:
resumo = pd.DataFrame(
    {
        "bruto": [
            df_bruto.shape[0],
            df_bruto.shape[1],
            df_bruto.isna().sum().sum(),
            df_bruto.duplicated(subset=["track_name", "artist(s)_name"]).sum(),
        ],
        "limpo": [
            df.shape[0],
            df.shape[1],
            df.isna().sum().sum(),
            df.duplicated(subset=["track_name", "artist_name"]).sum(),
        ],
    },
    index=["linhas", "colunas", "valores faltantes", "duplicatas (música + artista)"],
)
resumo

,bruto,limpo
linhas,953,949
colunas,24,21
valores faltantes,145,2
duplicatas (música + artista),4,0


In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 949 entries, 0 to 952
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   track_name            949 non-null    str           
 1   artist_name           949 non-null    str           
 2   artist_count          949 non-null    int64         
 3   released_year         949 non-null    int64         
 4   released_month        949 non-null    int64         
 5   released_day          949 non-null    int64         
 6   streams               947 non-null    float64       
 7   in_spotify_playlists  949 non-null    int64         
 8   in_spotify_charts     949 non-null    int64         
 9   bpm                   949 non-null    int64         
 10  key                   949 non-null    str           
 11  mode                  949 non-null    str           
 12  danceability          949 non-null    int64         
 13  valence               949 non-null  

In [21]:
print(df.isnull().sum())

track_name              0
artist_name             0
artist_count            0
released_year           0
released_month          0
released_day            0
streams                 2
in_spotify_playlists    0
in_spotify_charts       0
bpm                     0
key                     0
mode                    0
danceability            0
valence                 0
energy                  0
acousticness            0
instrumentalness        0
liveness                0
speechiness             0
data_lancamento         0
data_suspeita           0
dtype: int64


**Sobre os dois valores nulos na col `streams`:** Optei por mantê-los, pois ainda é possível utilizar outros parâmetros da linha, como in_spotify_playlists e in_spotify_charts. Além disso, os valores NaN não atrapalham os cálculos feitos pelo Pandas. 

## Salvamento
O dataset limpo é salvo em `data/spotify_2023_limpo.csv` para uso no notebook 03. O índice original
é preservado na coluna `id_original`, permitindo rastrear cada linha até o dataset bruto.

In [22]:
salvar_dados(df, "../data/spotify_2023_limpo.csv")

Dados salvos em ../data/spotify_2023_limpo.csv: 949 linhas e 21 colunas.
